The goal is understand:

**What information do I need from my LLM application so that I can understand what happened, why it happened, and how the system is performing?**

##### What is Observability?
Observability is the ability to understand the internal behaviour of a system by examining the information it produces.

That information is primarily:
* Logs
* metrics
* Traces

OpenTelemetry defines observability around telemetry such as traces, metrics, and logs, and describes instrumentation as the mechanism by which applications emit that telemetry.

1.1 Simple real-world example

Imagine your LangGraph application receives this request:

``` markdown
User:
"Compare Q1 and Q2 revenue by region."
``

Application Excecutes: 
``` markdown
User
 ↓
FastAPI
 ↓
LangGraph Supervisor
 ↓
SQL Agent
 ↓
LLM
 ↓
SQL Tool
 ↓
Database
 ↓
Analyst Agent
 ↓
LLM
 ↓
Final Answer
```
``` markdown
The user only sees:
"South region had the highest growth."
```

But as an AI Engineer, you need to know:

``` markdown
How long did it take?
How many LLM calls happened?
Which model was used?
How many tokens?
How much did it cost?
Which agent was selected?
Which SQL query was generated?
Did the SQL execute successfully?
What data came back?
Did the analyst interpret it correctly?
Did anything fail?
```
This is what observability gives you

##### Observability vs Monitoring

These two terms are often confused. they are related, but they are not the same thing.

**Monitoring** 

Monitoring usually asks """is something wrong"""

for example : 
``` markdown 
Error rate = 8% 
here we might have an alert :
ERROR RATE > 5%
        ↓
      ALERT
```

Here monitoring tells you there is a problem.

**Observability**

Observability asks **why is this happening**
``` markdown
suppose the error rate:

Error rate = 8%
```

Observability allows us to investigate

``` markdown
Request
 ↓
Supervisor
 ↓
SQL Agent
 ↓
LLM
 ↓
Generated SQL
 ↓
Database
 ↓
SQL Error
```

After all these investigations we got to know the root cause.

``` markdown
Monitoring → Detection

Observability → Investigation + Understanding
```

#### The Three Pillars

The classic observability model is:

``` markdown
                 OBSERVABILITY
                      │
          ┌───────────┼───────────┐
          ↓           ↓           ↓
        LOGS        METRICS     TRACES
```

| Signal  | Main question                                 |
| ------- | --------------------------------------------- |
| Logs    | What happened?                                |
| Metrics | How is the system behaving overall?           |
| Traces  | What happened during this particular request? |


1. Logs

A log is a record of something that happened in the application.

In [1]:
## Example :
print("Sql Agent Started")


Sql Agent Started


That's technically output, but in production we generally use Python's logging module.

In [9]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logging.info("SQL Agent Started")

INFO:root:SQL Agent Started


##### Different Log Levels

``` markdown
DEBUG
INFO
WARNING
ERROR
CRITICAL
```

DEBUG

Detailed information useful during development.

```python
logger.debug("Retrieved 5 documents")
```

INFO

Normal Application Activity

```python
logger.info("SQL agent started")
```

WARNING

Something unexpected happened, but application can continue.

```python 
logger.warning("Retriever returned only 1 document")
```

ERROR : Something Failed

```python 
logger.error("SQL execution failed")
```

CRITICAL : Serious Failure
``` python
logger.critical("Database service unavailable")
```



In [10]:
#### Basic LLM Application Logging

import logging
import time

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)


def ask_llm(question: str):

    logger.info("LLM request started")

    start_time = time.perf_counter()

    # Simulating LLM call
    time.sleep(1)

    answer = "Machine learning allows systems to learn from data."

    latency = time.perf_counter() - start_time

    logger.info(
        "LLM request completed | latency=%.2fs",
        latency
    )

    return answer


answer = ask_llm(
    "What is machine learning?"
)

print(answer)

INFO:__main__:LLM request started
INFO:__main__:LLM request completed | latency=1.00s


Machine learning allows systems to learn from data.


simple logs are not enough why becuase 

suppose in production system : 1lakh requests 

and our log files contains :

``` markdown
LLM request started
LLM request completed
LLM request started
LLM request completed
...
```

So here how do we answer which logs belongs to request #83792? for this case we need a request id.

#### Request ID / Correlation ID

A common pattern:
``` mrakdown
request id = abc 123
Then every log related to that request contains:
abc123
```

for example:

``` markdown
INFO | abc123 | Supervisor started
INFO | abc123 | SQL agent started
INFO | abc123 | SQL tool called
INFO | abc123 | SQL execution completed
INFO | abc123 | Final answer generated
```



Practical Request ID example

In [11]:
import logging
import uuid

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)


def process_request(question):

    request_id = str(uuid.uuid4())

    logger.info(
        "[%s] Request started",
        request_id
    )

    logger.info(
        "[%s] Running agent",
        request_id
    )

    logger.info(
        "[%s] Request completed",
        request_id
    )


process_request(
    "What was our Q1 revenue?"
)

INFO:__main__:[e810a087-38ba-4e31-ad6e-2b5f7f57447f] Request started
INFO:__main__:[e810a087-38ba-4e31-ad6e-2b5f7f57447f] Running agent
INFO:__main__:[e810a087-38ba-4e31-ad6e-2b5f7f57447f] Request completed


#### What should you log in an LLM application?

For an LLM application, useful logs include:

``` markdown
Request ID
User/session ID where appropriate
Agent name
Node name
Tool name
Model name
Environment
Success/failure
Error type
Latency
Token counts
Retrieved document count
Tool execution status
```

Note : Do not blindly log sensitive prompts, responses, credentials, PII, API keys, or confidential documents

#### Second Pillar — Metrics

logs tell you individual events.

Metrics tell you what is happening across many requests.

A metric is a numerical measurement captured over time. OpenTelemetry describes metrics as runtime measurements useful for understanding application availability and performance.

Example 

Suppose LLM Application recieves 1lakh + requests.

we dont want to inspect 10k logs , instead we calculate :

``` markdown
Total requests = 10,000
Successful = 9,700
Failed = 300
Error rate = 3%
Average latency = 2.4 sec
P95 latency = 5.1 sec
```
These are the metrics

##### Common Application Metrics

1. Request count --> requests_total
2. Error count --> erros_total
3. Error rate --> errors / total requests
4. Latency --> average latency, P50, P95, P99
5. Throuhput --> requests / second


for llm applications additionally:
``` markdown
input tokens
ouput tokens
total tokens
LLM calls
tool calls
retriver calls 
cost
```


Why P95 and P99 matter

suppose for 100 requests:

latency:

``` markdown
98 requests → 1 second
2 requests  → 20 seconds
```

Average might still look reasonable.
But some users are experiencing terrible latency.
That's why we use percentiles.

P50

50% of requests are faster than this.

P95

95% of requests are faster than this.

P99

99% of requests are faster than this.

For production systems, P95/P99 can be more useful than averages.

Practical latency metric example

In [12]:
import time
import random

latencies = []

for i in range(20):

    start = time.perf_counter()

    # Simulate request
    time.sleep(random.uniform(0.1, 0.8))

    latency = time.perf_counter() - start

    latencies.append(latency)


latencies.sort()

p50 = latencies[int(len(latencies) * 0.50)]
p95 = latencies[int(len(latencies) * 0.95)]

print(f"P50 latency: {p50:.2f}s")
print(f"P95 latency: {p95:.2f}s")

P50 latency: 0.53s
P95 latency: 0.77s


##### LLM-specific metrics

``` markdown
Request #1
Input tokens = 1000
Output tokens = 300

Request #2
Input tokens = 5000
Output tokens = 900

Request #3
Input tokens = 1200
Output tokens = 400
```

then we can calculate:
``` markdown
Total input tokens
Total output tokens
Average tokens/request
Tokens/request by model
```
Then cost:

``` markdown
Input tokens
+
Output tokens
+
model pricing
```
Later, when we study LLMOps Performance and Cost Engineering, these become critical.


#### Third Pillar — Traces

A Trace represents the complete journey of one request through the system.

OpenTelemetry describes a trace as one or more spans, with the root span representing the request and child spans representing operations within it.

``` markdown 
REQUEST
   │
   └── TRACE
        │
        ├── SPAN
        ├── SPAN
        ├── SPAN
        └── SPAN
```

A Trace is the whole story.

A Span is one chapter of that story.

example : suppose user asks : ``` what was q1 revenue?```

application

``` markdown
Request
 ↓
Supervisor
 ↓
SQL Agent
 ↓
LLM
 ↓
SQL Tool
 ↓
Database
 ↓
Final Answer
```

The trace could look like:

``` markdown

Trace: abc123

├── Supervisor
│
├── SQL Agent
│   │
│   ├── LLM Call
│   │
│   └── SQL Tool
│       │
│       └── Database
│
└── Final Response
```

Each operation becomes a span.

#### Trace vs Span

Trace represents one complete request/workflow. 

Span represents one operation within that request.

Think like this 

``` markdown 
Trace = Movie
Span = Scene

or 

Trace = Journey

Span = Individual stops
```

Observability systems commonly visualize traces as waterfall diagrams. OpenTelemetry's observability documentation describes this parent-child structure and waterfall representation.

``` markdown
0s       1s       2s       3s       4s

Request  ├──────────────────────────────┤

Router   ├───┤

LLM             ├─────────────┤

Retriever       ├────┤

Vector DB          ├──┤

Tool                    ├──────┤

Final                           ├──┤
```

Immediately we can see:
The LLM call is taking the most time.
This is why tracing is so powerful.

#### Nested spans
Spans can have parent-child relationships.

```markdown
Trace
│
└── Supervisor
     │
     ├── SQL Agent
     │    │
     │    ├── LLM
     │    │
     │    └── SQL Tool
     │         │
     │         └── Database
     │
     └── Final Answer
```

This hierarchy tells you how the request flowed through the application.
OpenTelemetry supports creating nested spans and tracking parent-child relationships.

Practical Python tracing — without any LLM

OpenTelemetry Python currently provides APIs/SDKs for traces and metrics, and its Python documentation supports manual instrumentation.

In [13]:
pip install opentelemetry-api opentelemetry-sdk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
## Create a First span
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor
)


# Create provider
provider = TracerProvider()

# Send spans to console
processor = SimpleSpanProcessor(
    ConsoleSpanExporter()
)

provider.add_span_processor(processor)

# Register provider
trace.set_tracer_provider(provider)

# Create tracer
tracer = trace.get_tracer("my-llm-app")


def process_request():

    with tracer.start_as_current_span("process_request"):

        print("Processing request...")


process_request()

Processing request...
{
    "name": "process_request",
    "context": {
        "trace_id": "0x99d9c3733e574d0ec249cacaa71ed0eb",
        "span_id": "0x3dadd2d061371739",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-08-09T08:35:40.523093Z",
    "end_time": "2026-08-09T08:35:40.523297Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "fe8e00f4-32f4-4916-9019-2b83afc6bfe3",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [15]:
def call_llm():

    with tracer.start_as_current_span("llm_call"):

        print("Calling LLM...")

In [16]:
def process_request():

    with tracer.start_as_current_span("process_request"):

        print("Request received")

        call_llm()


process_request()

Request received
Calling LLM...
{
    "name": "llm_call",
    "context": {
        "trace_id": "0x3bf55c0e37886ad4c43a66cdf22253ba",
        "span_id": "0xe3aadd34d84bd325",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x05b67e3bca1bc242",
    "start_time": "2026-08-09T08:36:15.053366Z",
    "end_time": "2026-08-09T08:36:15.053397Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "fe8e00f4-32f4-4916-9019-2b83afc6bfe3",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "process_request",
    "context": {
        "trace_id": "0x3bf55c0e37886ad4c43a66cdf22253ba",
        "span_id": "0x05b67e3bca1bc242",
        "t

In [17]:
### Adding attributes to the span
def call_llm():

    with tracer.start_as_current_span("llm_call") as span:

        span.set_attribute(
            "llm.model",
            "example-model"
        )

        span.set_attribute(
            "llm.temperature",
            0.2
        )

        print("Calling LLM...")

In [18]:
import time

from opentelemetry import trace


def router():

    with tracer.start_as_current_span("router") as span:

        span.set_attribute(
            "agent.name",
            "supervisor"
        )

        time.sleep(0.2)


def retriever():

    with tracer.start_as_current_span("retriever") as span:

        span.set_attribute(
            "retrieval.top_k",
            5
        )

        time.sleep(0.3)


def llm_call():

    with tracer.start_as_current_span("llm_call") as span:

        span.set_attribute(
            "llm.model",
            "example-model"
        )

        time.sleep(0.8)


def tool_call():

    with tracer.start_as_current_span("tool_call") as span:

        span.set_attribute(
            "tool.name",
            "sql_database"
        )

        time.sleep(0.4)


def process_request():

    with tracer.start_as_current_span("request"):

        router()

        retriever()

        llm_call()

        tool_call()


process_request()

{
    "name": "router",
    "context": {
        "trace_id": "0x10656500ab4d2b75499deb74f29a020e",
        "span_id": "0x3c5165a4e6b05f9d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x79a67cd03c67b7a7",
    "start_time": "2026-08-09T08:37:31.154757Z",
    "end_time": "2026-08-09T08:37:31.355332Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "agent.name": "supervisor"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "fe8e00f4-32f4-4916-9019-2b83afc6bfe3",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "retriever",
    "context": {
        "trace_id": "0x10656500ab4d2b75499deb74f29a020e",
        "span_id": "0x1ee6918f61fcd345",
        "t

without tracing : Applicatio took 1.7 seconds.

with tracing:

``` markdown
Request
│
├── Router       0.2 sec
├── Retriever    0.3 sec
├── LLM          0.8 sec
└── Tool         0.4 sec
```


Now if we observe llm is biggest contributor thats observability.

#### Events

An event represents something that happened at a particular point in an operation.

for example:

``` markdown
LLM started
Tool selected
Documents retrieved
Validation failed
Retry started
```



In [19]:
with tracer.start_as_current_span("llm_call") as span:

    span.add_event(
        "prompt_prepared"
    )

    # LLM call

    span.add_event(
        "response_received"
    )

{
    "name": "llm_call",
    "context": {
        "trace_id": "0xf10aa0211618db8e2d33acd0d52c08b5",
        "span_id": "0xa619c6b39b8a4ade",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-08-09T08:40:37.147351Z",
    "end_time": "2026-08-09T08:40:37.147419Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [
        {
            "name": "prompt_prepared",
            "timestamp": "2026-08-09T08:40:37.147388Z",
            "attributes": {}
        },
        {
            "name": "response_received",
            "timestamp": "2026-08-09T08:40:37.147407Z",
            "attributes": {}
        }
    ],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "fe8e00f4-32f4-4916-9019-2b83afc6bfe3",
 

conceptually 

``` markdown
LLM span
│
├── Event: prompt_prepared
│
├── LLM execution
│
└── Event: response_received
```



All putting together
``` markdown
                OBSERVABILITY
                     │
       ┌─────────────┼─────────────┐
       ↓             ↓             ↓
     Logs         Metrics        Traces
       │             │             │
   Individual      Overall       Individual
    events        behavior       request
       │             │             │
       └─────────────┼─────────────┘
                     ↓
               Understand system
```

#### LLM-Specific Observability

for an LLM Call:
``` markdown
Model
Provider
Prompt
Response
Input tokens
Output tokens
Latency
Temperature
Max tokens
Finish reason
Error
Cost
```

for RAG operation:

``` markdown
Query
Retriever
Top K
Documents retrieved
Similarity scores
Latency
```

For an Agent

``` markdown
Agent name
Node
Decision
Tool selected
Tool arguments
Tool result
Iterations
State
```

conceptually for an llm application span:

``` markdown
Span: LLM Call

model = gpt-...
provider = ...
temperature = 0.2

input_tokens = 2100
output_tokens = 450

latency = 2.4 sec

status = success
```
Then

``` markdown
Span: Retriever

top_k = 5
documents = 5
latency = 0.35 sec
```

and Span tool

``` markdown
Span: Tool

tool = SQL
latency = 0.6 sec
status = success
```


#### The complete LLM trace

``` markdown
REQUEST
│
└── TRACE: request_123
     │
     ├── SPAN: FastAPI request
     │
     ├── SPAN: Supervisor
     │
     ├── SPAN: Router
     │
     ├── SPAN: Retriever
     │    │
     │    └── SPAN: Vector DB
     │
     ├── SPAN: LLM
     │    ├── model
     │    ├── tokens
     │    ├── latency
     │    └── cost
     │
     ├── SPAN: Tool
     │
     ├── SPAN: Database
     │
     └── SPAN: Final response
```